# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NourhanFarag/nourhan-flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

### Baseline Scoring Strategy

**Lane:** Refresh / Content Opportunity Scoring

**Unit of analysis:** One content page.

**Prediction cutoff:** March 14, 2026.

**Historical feature window:** March 1–14, 2026.

### Rule in Plain Words

Prioritize pages that already receive meaningful search
impressions but have low historical CTR.

Higher search exposure means a larger potential opportunity,
while sufficient observation days increase confidence in
the measurement.

The score supports editorial review, not automatic refresh.

### Scoring Rules

| Condition | Points |
|---|---:|
| 500+ impressions and CTR < 0.5% | +60 |
| 100–499 impressions and CTR < 0.5% | +30 |
| 2000+ impressions and CTR < 0.5% | +20 |
| 7+ GSC-observed days, 100+ impressions, CTR < 0.5% | +10 |

Maximum score: 90 points.

These are fixed, hand-selected heuristic weights,
not learned model coefficients or probabilities.

### Reason Codes

- `low_ctr_high_visibility`: High historical impressions with low CTR.
- `low_ctr_moderate_visibility`: Moderate historical impressions with low CTR.
- `large_search_exposure`: Additional opportunity from high impression volume.
- `sufficient_history`: At least seven GSC-observed days.
- `limited_history`: Fewer than seven observed days.
- `low_search_volume`: Fewer than 100 impressions.
- `no_gsc_data`: No observed historical GSC data.
- `not_flagged`: No low-CTR opportunity detected by this rule.

### Action Policy

- Score >= 60: Prioritize manual CTR and search-intent review.
- Score 30–59: Investigate when reviewer capacity allows.
- Score = 0: Monitor or mark insufficient evidence.

Missing GSC data is never interpreted as zero CTR.

Content age receives no weight because ML-06 found mixed evidence.

Average position receives no weight because its encoding
remains uncertain.

The ML-06 low-CTR flag test supports this direction,
but its results came from a more restricted cohort.
They do not establish the accuracy of this full baseline.

Future-window metrics and target-derived fields are
excluded from scoring.

In [1]:
# Fixed rules only. No model fitting.

import pandas as pd
import numpy as np

LOW_CTR_THRESHOLD = 0.5       # Percent, not fraction
MIN_IMPRESSIONS = 100
HIGH_IMPRESSIONS = 500
VERY_HIGH_IMPRESSIONS = 2000
MIN_GSC_DAYS = 7

SCORING_WEIGHTS = {
    "low_ctr_high_visibility": 60,
    "low_ctr_moderate_visibility": 30,
    "large_search_exposure": 20,
    "sufficient_history": 10,
}

assert 0 < MIN_IMPRESSIONS < HIGH_IMPRESSIONS
assert HIGH_IMPRESSIONS < VERY_HIGH_IMPRESSIONS
assert 0 < LOW_CTR_THRESHOLD < 100
assert 1 <= MIN_GSC_DAYS <= 14

rules_summary = pd.DataFrame([
    {
        "Reason Code": reason,
        "Points": points
    }
    for reason, points in SCORING_WEIGHTS.items()
])

print("ML-07 — Baseline Configuration")
display(rules_summary)

print("Low CTR threshold:", LOW_CTR_THRESHOLD, "%")
print("Minimum impressions:", MIN_IMPRESSIONS)
print("High visibility:", HIGH_IMPRESSIONS)
print("Very high visibility:", VERY_HIGH_IMPRESSIONS)
print("Minimum observed GSC days:", MIN_GSC_DAYS)

print("\nMaximum baseline score:", 90)
print("Configuration checks passed.")

ML-07 — Baseline Configuration


,Reason Code,Points
0,low_ctr_high_visibility,60
1,low_ctr_moderate_visibility,30
2,large_search_exposure,20
3,sufficient_history,10


Low CTR threshold: 0.5 %
Minimum impressions: 100
High visibility: 500
Very high visibility: 2000
Minimum observed GSC days: 7

Maximum baseline score: 90
Configuration checks passed.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [2]:
# Historical data only. No future features or model training.

import os
import getpass
from pathlib import Path
import duckdb
import pandas as pd
import numpy as np

# --------------------------------------------------
# 1. Secure warehouse connection
# --------------------------------------------------

HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    HF_TOKEN = getpass.getpass("Hugging Face READ token: ")

con = duckdb.connect()

safe_token = HF_TOKEN.replace("'", "''")

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{safe_token}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

MARCH_DAILY = (
    f"read_parquet("
    f"'{REL}/fact_content_daily_performance/month=2026-03/*.parquet'"
    f")"
)

DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

# --------------------------------------------------
# 2. Build historical-only feature frame
# --------------------------------------------------

baseline_frame = con.sql(f"""
WITH historical AS (
    SELECT
        client_hash_id,
        content_hash_id,

        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS prev_available_days,

        SUM(
            CASE
                WHEN gsc_data_available IS TRUE
                THEN COALESCE(gsc_impressions, 0)
                ELSE 0
            END
        ) AS impressions,

        SUM(
            CASE
                WHEN gsc_data_available IS TRUE
                THEN COALESCE(gsc_clicks, 0)
                ELSE 0
            END
        ) AS clicks

    FROM {MARCH_DAILY}

    WHERE report_date BETWEEN DATE '2026-03-01'
                          AND DATE '2026-03-14'

    GROUP BY client_hash_id, content_hash_id
)

SELECT
    h.client_hash_id,
    h.content_hash_id,

    CASE
        WHEN h.prev_available_days > 0
        THEN h.impressions
    END AS prev_14d_impressions,

    CASE
        WHEN h.impressions > 0
        THEN 100.0 * h.clicks / h.impressions
    END AS prev_14d_ctr,

    h.prev_available_days,

    DATE_DIFF(
        'day',
        c.content_created_date,
        DATE '2026-03-14'
    ) AS content_age_days

FROM historical AS h

INNER JOIN {DIM_CONTENT} AS c
    ON h.content_hash_id = c.content_hash_id

WHERE c.content_created_date IS NOT NULL
  AND c.content_created_date <= DATE '2026-03-14'
""").df()

assert not baseline_frame[
    ["client_hash_id", "content_hash_id"]
].duplicated().any()

assert baseline_frame["content_age_days"].ge(0).all()

print("Baseline frame:", baseline_frame.shape)

# --------------------------------------------------
# 3. Apply rule-based scoring
# --------------------------------------------------

imp = baseline_frame["prev_14d_impressions"]
ctr = baseline_frame["prev_14d_ctr"]
days = baseline_frame["prev_available_days"]

low_ctr = (
    imp.ge(MIN_IMPRESSIONS)
    & ctr.lt(LOW_CTR_THRESHOLD)
)

moderate_visibility = (
    low_ctr
    & imp.lt(HIGH_IMPRESSIONS)
)

high_visibility = (
    low_ctr
    & imp.ge(HIGH_IMPRESSIONS)
)

large_exposure = (
    low_ctr
    & imp.ge(VERY_HIGH_IMPRESSIONS)
)

sufficient_history = (
    low_ctr
    & days.ge(MIN_GSC_DAYS)
)

baseline_frame["baseline_action_score"] = (
    30 * moderate_visibility.astype(int)
    + 60 * high_visibility.astype(int)
    + 20 * large_exposure.astype(int)
    + 10 * sufficient_history.astype(int)
)

# --------------------------------------------------
# 4. Assign explainable reason codes
# --------------------------------------------------

baseline_frame["reason_codes"] = np.select(
    [
        imp.isna(),
        imp.lt(MIN_IMPRESSIONS),
        moderate_visibility,
        high_visibility,
    ],
    [
        "no_gsc_data",
        "low_search_volume",
        "low_ctr_moderate_visibility",
        "low_ctr_high_visibility",
    ],
    default="not_flagged",
)

baseline_frame.loc[
    large_exposure, "reason_codes"
] += "|large_search_exposure"

baseline_frame.loc[
    sufficient_history, "reason_codes"
] += "|sufficient_history"

baseline_frame.loc[
    low_ctr & days.lt(MIN_GSC_DAYS),
    "reason_codes"
] += "|limited_history"

# --------------------------------------------------
# 5. Assign suggested actions
# --------------------------------------------------

score = baseline_frame["baseline_action_score"]

baseline_frame["suggested_action"] = np.select(
    [
        score.ge(60),
        score.ge(30),
    ],
    [
        "Prioritize manual CTR and search-intent review",
        "Investigate CTR opportunity",
    ],
    default="Monitor or insufficient evidence",
)

# --------------------------------------------------
# 6. Rank all eligible historical pages
# --------------------------------------------------

ranked_queue = (
    baseline_frame
    .sort_values(
        by=[
            "baseline_action_score",
            "prev_14d_impressions",
            "prev_14d_ctr",
            "prev_available_days",
            "client_hash_id",
            "content_hash_id",
        ],
        ascending=[
            False, False, True, False, True, True
        ],
        na_position="last",
        kind="mergesort",
    )
    .reset_index(drop=True)
)

ranked_queue.insert(
    0, "rank",
    np.arange(1, len(ranked_queue) + 1)
)

# --------------------------------------------------
# 7. Validate results
# --------------------------------------------------

assert ranked_queue["baseline_action_score"].between(
    0, 90
).all()

assert ranked_queue["baseline_action_score"].is_monotonic_decreasing

assert ranked_queue.loc[
    ranked_queue["prev_14d_impressions"].isna(),
    "baseline_action_score"
].eq(0).all()

assert ranked_queue["rank"].is_unique

# --------------------------------------------------
# 8. Save locally — do not commit full warehouse data
# --------------------------------------------------

current = Path.cwd().resolve()

repo_root = next(
    (
        p for p in [current, *current.parents]
        if (p / "work" / "notebooks"
            / "w04_baseline_score.ipynb").exists()
    ),
    None
)

if repo_root is None:
    raise FileNotFoundError(
        "Repository root not found. Check your notebook location."
    )

output_dir = repo_root / "work" / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "baseline_action_score.csv"

ranked_queue.to_csv(output_path, index=False)

# --------------------------------------------------
# 9. Safe aggregate reporting
# --------------------------------------------------

score_distribution = (
    ranked_queue.groupby("baseline_action_score")
    .size()
    .reset_index(name="pages")
    .sort_values("baseline_action_score", ascending=False)
)

print("\nML-07 — Baseline Score Distribution")
display(score_distribution)

print("Pages ranked:", len(ranked_queue))

print(
    "Pages with nonzero score:",
    int(ranked_queue["baseline_action_score"].gt(0).sum())
)

print("Maximum score:", ranked_queue["baseline_action_score"].max())
print("CSV saved to:", output_path)
print("Baseline validation passed.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Baseline frame: (317443, 6)

ML-07 — Baseline Score Distribution


,baseline_action_score,pages
6,90,11742
5,80,28
4,70,20489
3,60,155
2,40,27375
1,30,802
0,0,256852


Pages ranked: 317443
Pages with nonzero score: 60591
Maximum score: 90
CSV saved to: E:\FlyRank\nourhan-flyrank-ml-internship\work\outputs\baseline_action_score.csv
Baseline validation passed.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [3]:
# Historical evidence only; no future metrics.

top20_review = ranked_queue.head(20).copy()

# Confidence refers to measurement coverage,
# NOT certainty that refreshing will improve performance.

days = top20_review["prev_available_days"]

top20_review["confidence_note"] = np.select(
    [
        days.ge(12),
        days.ge(7),
        days.gt(0),
    ],
    [
        "High measurement coverage; action unverified",
        "Moderate measurement coverage; action unverified",
        "Limited measurement coverage; action unverified",
    ],
    default="Insufficient GSC observations",
)

# Possible failure explanations, not verified causes.
top20_review["what_could_make_it_wrong"] = np.select(
    [
        top20_review["prev_14d_ctr"].eq(0),
        days.lt(10),
    ],
    [
        "Zero clicks may reflect query intent or SERP context, not a content problem",
        "Limited observed days may make the CTR estimate unstable",
    ],
    default=(
        "Low CTR may be normal for the query mix; "
        "a content refresh may not improve it"
    ),
)

top20_review["manual_review_status"] = (
    "Needs editorial/context verification"
)

review_columns = [
    "rank",
    "baseline_action_score",
    "prev_14d_impressions",
    "prev_14d_ctr",
    "prev_available_days",
    "suggested_action",
    "reason_codes",
    "confidence_note",
    "what_could_make_it_wrong",
    "manual_review_status",
]

print("ML-07 — Top-20 Review")
display(top20_review[review_columns].round(2))

print("\nTop-20 Summary")
print("Pages reviewed numerically:", len(top20_review))
print(
    "Unique clients represented:",
    top20_review["client_hash_id"].nunique()
)
print(
    "Score range:",
    top20_review["baseline_action_score"].min(),
    "to",
    top20_review["baseline_action_score"].max()
)
print(
    "Pages with 12+ observed days:",
    int(top20_review["prev_available_days"].ge(12).sum())
)

assert len(top20_review) == 20
assert top20_review["baseline_action_score"].is_monotonic_decreasing

print("\nTop-20 review table generated successfully.")

ML-07 — Top-20 Review


,rank,baseline_action_score,prev_14d_impressions,prev_14d_ctr,prev_available_days,suggested_action,reason_codes,confidence_note,what_could_make_it_wrong,manual_review_status
0,1,90,138793.0,0.24,14,Prioritize manual CTR and search-intent review,low_ctr_high_visibility|large_search_exposure|...,High measurement coverage; action unverified,Low CTR may be normal for the query mix; a con...,Needs editorial/context verification
1,2,90,105148.0,0.10,14,Prioritize manual CTR and search-intent review,low_ctr_high_visibility|large_search_exposure|...,High measurement coverage; action unverified,Low CTR may be normal for the query mix; a con...,Needs editorial/context verification
2,3,90,102917.0,0.44,14,Prioritize manual CTR and search-intent review,low_ctr_high_visibility|large_search_exposure|...,High measurement coverage; action unverified,Low CTR may be normal for the query mix; a con...,Needs editorial/context verification
3,4,90,86624.0,0.20,14,Prioritize manual CTR and search-intent review,low_ctr_high_visibility|large_search_exposure|...,High measurement coverage; action unverified,Low CTR may be normal for the query mix; a con...,Needs editorial/context verification
4,5,90,84160.0,0.06,14,Prioritize manual CTR and search-intent review,low_ctr_high_visibility|large_search_exposure|...,High measurement coverage; action unverified,Low CTR may be normal for the query mix; a con...,Needs editorial/context verification
5,6,90,83770.0,0.00,14,Prioritize manual CTR and search-intent review,low_ctr_high_visibility|large_search_exposure|...,High measurement coverage; action unverified,Zero clicks may reflect query intent or SERP c...,Needs editorial/context verification
6,7,90,81456.0,0.14,14,Prioritize manual CTR and search-intent review,low_ctr_high_visibility|large_search_exposure|...,High measurement coverage; action unverified,Low CTR may be normal for the query mix; a con...,Needs editorial/context verification
7,8,90,77413.0,0.16,14,Prioritize manual CTR and search-intent review,low_ctr_high_visibility|large_search_exposure|...,High measurement coverage; action unverified,Low CTR may be normal for the query mix; a con...,Needs editorial/context verification
8,9,90,75458.0,0.26,14,Prioritize manual CTR and search-intent review,low_ctr_high_visibility|large_search_exposure|...,High measurement coverage; action unverified,Low CTR may be normal for the query mix; a con...,Needs editorial/context verification
9,10,90,73817.0,0.26,14,Prioritize manual CTR and search-intent review,low_ctr_high_visibility|large_search_exposure|...,High measurement coverage; action unverified,Low CTR may be normal for the query mix; a con...,Needs editorial/context verification



Top-20 Summary
Pages reviewed numerically: 20
Unique clients represented: 4
Score range: 90 to 90
Pages with 12+ observed days: 20

Top-20 review table generated successfully.


### Top-20 Review Method

The top 20 pages were selected from the historical-only
ranked baseline queue.

For each candidate, the review records:
- A suggested manual action.
- The rule-based reason codes.
- Measurement-coverage confidence.
- A potential explanation for why the selection could be wrong.

Confidence describes the completeness of historical
observations, not the probability of refresh success.

The failure explanations are hypotheses requiring
editorial verification, not observed page-level causes.

Since page titles, query intent, and SERP context are
not available in the audit frame, a definitive editorial
judgment cannot be made from these metrics alone.

The review does not use future performance, model
predictions, or FlyRank product decision flags.

## 4. Weak picks + leakage check

### Weak Picks and Baseline Limitations

The top 20 candidates all received a score of 90 and
had 14 days of historical GSC coverage.

However, high measurement coverage does not prove that
a page needs refreshing.

#### Potential Weak Pick 1 — Rank 3

- Historical impressions: 102,917.
- Historical CTR: approximately 0.44%.
- Score: 90.
- Weakness: CTR is close to the 0.5% threshold.
- Potential issue: A small difference in CTR could change
  eligibility without necessarily changing editorial value.

#### Potential Weak Pick 2 — Rank 6

- Historical impressions: 83,770.
- Historical CTR: 0%.
- Score: 90.
- Weakness: Zero clicks alone does not establish that
  refreshing the content is the appropriate action.
- Potential issue: Query intent, search-result features,
  or other factors could explain the low CTR.

These are potential weaknesses, not confirmed bad picks.

### Baseline Limitations

1. The score uses manually selected thresholds and weights.
2. Many pages share the maximum score of 90.
3. Tie-breaking prioritizes impression volume rather than
   estimated improvement or editorial impact.
4. Missing GSC data produces a zero score, representing
   insufficient evidence rather than healthy performance.
5. Client concentration may reduce queue diversity.
6. The rule does not observe search intent, page quality,
   or whether a content change would improve results.
7. The baseline has not been validated against actual
   editorial outcomes.

### Leakage Policy

Scoring uses only:

- `prev_14d_impressions`
- `prev_14d_ctr`
- `prev_available_days`

All three are calculated from March 1–14, 2026.

Future-window performance and target-derived fields
are excluded from score construction.

Content identifiers are used only for joining and
deterministic tie-breaking, not score calculation.

This is a review-priority heuristic, not a probability
of content requiring refresh.

In [4]:
# ----------------------------------------------
# 1. Inspect potential weak picks
# ----------------------------------------------

weak_picks = top20_review.loc[
    top20_review["rank"].isin([3, 6]),
    [
        "rank",
        "baseline_action_score",
        "prev_14d_impressions",
        "prev_14d_ctr",
        "prev_available_days",
        "what_could_make_it_wrong",
    ]
]

print("ML-07 — Potential Weak Picks")
display(weak_picks)

# ----------------------------------------------
# 2. Verify scoring inputs
# ----------------------------------------------

scoring_inputs = [
    "prev_14d_impressions",
    "prev_14d_ctr",
    "prev_available_days",
]

forbidden_patterns = (
    "future_",
    "current_14d_",
    "needs_review",
    "target",
    "proxy",
)

assert all(
    not any(pattern in col for pattern in forbidden_patterns)
    for col in scoring_inputs
)

assert set(scoring_inputs).issubset(ranked_queue.columns)

# ----------------------------------------------
# 3. Recalculate scores independently
# ----------------------------------------------

imp = ranked_queue["prev_14d_impressions"]
ctr = ranked_queue["prev_14d_ctr"]
days = ranked_queue["prev_available_days"]

candidate = (
    imp.ge(MIN_IMPRESSIONS)
    & ctr.lt(LOW_CTR_THRESHOLD)
)

expected_score = (
    60 * (candidate & imp.ge(HIGH_IMPRESSIONS)).astype(int)
    + 30 * (
        candidate & imp.lt(HIGH_IMPRESSIONS)
    ).astype(int)
    + 20 * (
        candidate & imp.ge(VERY_HIGH_IMPRESSIONS)
    ).astype(int)
    + 10 * (
        candidate & days.ge(MIN_GSC_DAYS)
    ).astype(int)
)

score_matches = (
    expected_score
    .eq(ranked_queue["baseline_action_score"])
    .all()
)

assert score_matches

# ----------------------------------------------
# 4. Check missing-data behavior
# ----------------------------------------------

missing_gsc_score_zero = ranked_queue.loc[
    ranked_queue["prev_14d_impressions"].isna(),
    "baseline_action_score"
].eq(0).all()

assert missing_gsc_score_zero

# ----------------------------------------------
# 5. Check queue and top-20 consistency
# ----------------------------------------------

assert len(top20_review) == 20

assert top20_review["rank"].tolist() == list(range(1, 21))

assert ranked_queue["baseline_action_score"].is_monotonic_decreasing

# ----------------------------------------------
# 6. Final report
# ----------------------------------------------

print("\nML-07 — Final Validation")

print("Scoring inputs:", scoring_inputs)
print("Scores match rule:", bool(score_matches))
print("Missing GSC scores zero:", bool(missing_gsc_score_zero))
print("Top-20 pages:", len(top20_review))
print(
    "Unique clients in Top-20:",
    top20_review["client_hash_id"].nunique()
)

print("\nAll baseline validation checks passed.")

ML-07 — Potential Weak Picks


,rank,baseline_action_score,prev_14d_impressions,prev_14d_ctr,prev_available_days,what_could_make_it_wrong
2,3,90,102917.0,0.444047,14,Low CTR may be normal for the query mix; a con...
5,6,90,83770.0,0.000000,14,Zero clicks may reflect query intent or SERP c...



ML-07 — Final Validation
Scoring inputs: ['prev_14d_impressions', 'prev_14d_ctr', 'prev_available_days']
Scores match rule: True
Missing GSC scores zero: True
Top-20 pages: 20
Unique clients in Top-20: 4

All baseline validation checks passed.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.